In [1]:
import torch
from torch import nn

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Selected device:", device)

Selected device: cuda


In [2]:
torch.manual_seed(42)

x = torch.linspace(-1, 1, 100).unsqueeze(1)
y = 2 * x + 1

model = nn.Linear(1, 1).to(device)

x = x.to(device)
y = y.to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.05
)

loss_fn = nn.MSELoss()

print("Model device:", next(model.parameters()).device)
print("Input device:", x.device)
print("Target device:", y.device)

Model device: cuda:0
Input device: cuda:0
Target device: cuda:0


In [3]:
def train(model, x, y, loss_fn, optimizer, epochs=100):
    history = []

    model.train()

    for epoch in range(epochs):
        optimizer.zero_grad()

        y_pred = model(x)
        loss = loss_fn(y_pred, y)

        loss.backward()
        optimizer.step()

        history.append(loss.item())

    return history


history = train(
    model, x, y, loss_fn, optimizer, epochs=100
)

print("Final loss:", history[-1])

Final loss: 1.0682230140446336e-06


In [4]:
from pathlib import Path

checkpoint_dir = Path("checkpoints")
checkpoint_dir.mkdir(exist_ok=True)

checkpoint_path = checkpoint_dir / "day41_linear.pt"

checkpoint = {
    "epoch": 100,
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "loss": history[-1],
}

torch.save(checkpoint, checkpoint_path)

print("Checkpoint saved:", checkpoint_path)

Checkpoint saved: checkpoints\day41_linear.pt


In [6]:
model.eval()

with torch.no_grad():
    expected = model(x).detach().cpu()

restored_model = nn.Linear(1, 1).to(device)

restored_optimizer = torch.optim.Adam(
    restored_model.parameters(),
    lr=0.05
)

loaded = torch.load(
    checkpoint_path,
    map_location=device,
    weights_only=True
)

restored_model.load_state_dict(
    loaded["model_state_dict"]
)

restored_optimizer.load_state_dict(
    loaded["optimizer_state_dict"]
)

restored_model.eval()

with torch.no_grad():
    actual = restored_model(x).detach().cpu()

torch.testing.assert_close(actual, expected)

print("Checkpoint restored successfully!")
print("Restored epoch:", loaded["epoch"])
print("Restored loss:", loaded["loss"])

print("Optimizer state restored:",
      len(restored_optimizer.state) > 0)

Checkpoint restored successfully!
Restored epoch: 100
Restored loss: 1.0682230140446336e-06
Optimizer state restored: True
